In [ ]:
import os
from typing import Dict, Any
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_core.tools import tool
from langchain.agents import create_agent

import sys
sys.path.append(r"C:\My Projects\Health-Navigator")

from dotenv import load_dotenv
load_dotenv(r'C:\My Projects\Health-Navigator\credentials.env')

MEDICAL_AGENT_SYSTEM_PROMPT = """You are an Expert Medical AI Assistant providing comprehensive clinical analysis and evidence-based recommendations.

## Your Role:
Analyze patient information, formulate differential diagnoses, and provide medical guidance while identifying when additional information is critical for accurate assessment.

## Analysis Framework:
1. **Chief Complaint**: Understand primary concern
2. **Differential Diagnosis**: List conditions from most to least likely
3. **Risk Stratification**: Identify urgency level
4. **Information Adequacy**: Assess if you have sufficient data
5. **Recommendations**: Provide clear, actionable next steps

## When You Have Sufficient Information:

Provide structured response:

**CLINICAL ASSESSMENT:**
[Your detailed assessment of the situation]

**DIFFERENTIAL DIAGNOSES:**
1. [Most likely] - [Evidence: symptoms, findings, risk factors]
2. [Second possibility] - [Evidence]
3. [Other considerations] - [Evidence]

**RISK LEVEL:** Low/Moderate/High/Critical
[Justification for risk level]

**RECOMMENDATIONS:**
- **Immediate Actions:** [If any urgent steps needed]
- **Follow-up:** [Timeline and monitoring plan]
- **Lifestyle/Management:** [Relevant advice]
- **Red Flags:** [When to seek emergency care]

**CLINICAL REASONING:**
[Explain your diagnostic thinking, evidence basis, confidence level]

**MEDICAL DISCLAIMER:**
This is AI-assisted analysis. Always consult healthcare professionals for diagnosis and treatment. Seek immediate medical attention for emergencies.

## When You Need More Information:

If critical information is missing that affects diagnostic accuracy or safety:
```
NEED_MORE_INFO: [Specific information needed]

CLINICAL JUSTIFICATION: [Explain why this is critical for assessment and what it will help determine]
```

Examples of valid information requests:
- "Patient's age and gender - essential for interpreting lab values and risk stratification"
- "Duration and progression of symptoms - needed to differentiate acute vs. chronic"
- "Current medications - must check for drug interactions and contraindications"
- "Previous imaging for comparison - critical for assessing disease progression"

## Reflection Guidelines:
- Current attempt: {reflection_count}/{max_reflections}
- Only request CRITICAL information for safe/accurate assessment
- Work with available data when appropriate, noting limitations
- At max reflections, provide best assessment possible with caveats

## Professional Standards:
- Evidence-based medicine principles
- Patient safety is top priority
- Clear about certainty levels (definitive/possible/unlikely)
- Acknowledge limitations
- Use patient-friendly language while maintaining clinical accuracy
- Never provide definitive diagnoses - recommend professional confirmation

## Critical Conditions Requiring Extra Caution:
- Chest pain, severe headaches, neurological symptoms
- Children, elderly, pregnant patients
- Severe pain, bleeding, breathing difficulties
- Mental health crises

You are an AI assistant supporting healthcare decisions, not replacing healthcare professionals."""


llm = ChatGoogleGenerativeAI(
    model="gemini-3-pro-preview",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    )

def invoke_medical_agent(
    aggregated_output: str,
    db_results: str,
    reflection_count: int,
    max_reflections: int,
    conversation_history: list
) -> Dict[str, Any]:
    """
    Invokes the medical agent to analyze patient information.
    
    Returns:
        dict with keys: 'response', 'conversation_history', 'needs_more_info', 'info_request'
    """


    # Format system prompt
    formatted_system_prompt = MEDICAL_AGENT_SYSTEM_PROMPT.format(
        reflection_count=reflection_count,
        max_reflections=max_reflections
    )
    
    # Build comprehensive medical context
    medical_context = f"""
    PATIENT CASE INFORMATION:
    
    Initial Analysis Output:
    {aggregated_output}
    
    Database Query Results:
    {db_results}
    
    Task: Provide comprehensive medical assessment. If critical information is missing
    and you're below {max_reflections} reflections, request specific information using
    NEED_MORE_INFO format. Otherwise, provide best assessment with available data.
    """
    
    # Create medical agent (simple LLM call, no tools needed)
    messages = conversation_history + [
        SystemMessage(content=formatted_system_prompt),
        HumanMessage(content=medical_context)
    ]
    
    response = llm.invoke(messages)
    agent_response = response.content
    
    # Update conversation history
    updated_history = conversation_history + [response.content[0]['text']]
    
    # Check if agent needs more information
    needs_more_info = False
    info_request = ""
    
    if "NEED_MORE_INFO:" in agent_response and reflection_count < max_reflections:
        # Extract information request
        info_lines = agent_response.split("NEED_MORE_INFO:")[1].strip().split("\n")
        info_request = info_lines[0].strip()
        needs_more_info = True
    
    return {
        'response': agent_response,
        'conversation_history': updated_history,
        'needs_more_info': needs_more_info,
        'info_request': info_request
    }

In [3]:
# 2. Medical Agent call
medical_result = invoke_medical_agent(
    aggregated_output="""
    Input Prompt: Patient experiencing chest pain and shortness of breath
    
    Numerical Analysis:
    Input: Heart rate: 105 bpm, Blood pressure: 145/90
    Output: Elevated vital signs detected
    
    Vision Analysis:
    Input: ECG image uploaded
    Output: Possible ST elevation observed
    """,
    db_results="""
    INFORMATION_COMPLETE
    
    **VECTOR DATABASE RESULTS:**
    Patient has history of hypertension, diagnosed 2020. Previous ECG normal in 2023.
    
    **RELATIONAL DATABASE RESULTS:**
    Age: 58, Gender: Male
    Current medications: Lisinopril 10mg daily
    Last visit: 3 months ago for routine checkup
    
    **USER PROVIDED INFORMATION:**
    Pain started 2 hours ago, radiating to left arm
    
    **SUMMARY:** 58yo male with hypertension presenting with acute chest pain
    """,
    reflection_count=0,
    max_reflections=5,
    conversation_history=[]
)

In [ ]:
print(medical_result)

{'response': [{'type': 'text', 'text': '**CLINICAL ASSESSMENT:**\nThe patient is a 58-year-old male with a history of hypertension presenting with classic signs of Acute Coronary Syndrome (ACS). The chief complaints are acute chest pain (onset 2 hours ago) radiating to the left arm and shortness of breath. Vital signs demonstrate tachycardia (HR 105) and hypertension (145/90), consistent with sympathetic surge due to pain or cardiac distress. Most critically, the Vision Analysis of the ECG indicates possible ST elevation. This clinical constellation (symptoms + demographics + ECG findings) is highly suggestive of an acute ST-Elevation Myocardial Infarction (STEMI), which is a time-critical medical emergency.\n\n**DIFFERENTIAL DIAGNOSES:**\n1.  **Acute ST-Elevation Myocardial Infarction (STEMI)** - [Evidence: Acute chest pain radiating to left arm, onset <12 hours, tachycardia, and "Possible ST elevation" on ECG analysis.]\n2.  **Non-ST-Elevation Myocardial Infarction (NSTEMI) / Unstabl

In [6]:
data = medical_result

main_text = data['response'][0]['text']
signature = data['response'][0]['extras']['signature']
history = data['conversation_history']


In [7]:
print(main_text)

**CLINICAL ASSESSMENT:**
The patient is a 58-year-old male with a history of hypertension presenting with classic signs of Acute Coronary Syndrome (ACS). The chief complaints are acute chest pain (onset 2 hours ago) radiating to the left arm and shortness of breath. Vital signs demonstrate tachycardia (HR 105) and hypertension (145/90), consistent with sympathetic surge due to pain or cardiac distress. Most critically, the Vision Analysis of the ECG indicates possible ST elevation. This clinical constellation (symptoms + demographics + ECG findings) is highly suggestive of an acute ST-Elevation Myocardial Infarction (STEMI), which is a time-critical medical emergency.

**DIFFERENTIAL DIAGNOSES:**
1.  **Acute ST-Elevation Myocardial Infarction (STEMI)** - [Evidence: Acute chest pain radiating to left arm, onset <12 hours, tachycardia, and "Possible ST elevation" on ECG analysis.]
2.  **Non-ST-Elevation Myocardial Infarction (NSTEMI) / Unstable Angina** - [Evidence: History of hypertensi

In [10]:
print(signature)

Et8mCtwmAXLI2nyU92WZbmelsnChoOkzIKAigdL9hApwHwhfeeOyk+M4btJIA4Faz/66AWFSkfn4OdaOBMtSQT0gE8kqz6Rro8tOHBjcJRtZAUimmJyby7T7VJHAiBMVWu2G6wBXkbWu4DnbBQcB9i6GBdZyh6VBEQQl7YDrCR7Ngo4JikrwfssTvOuWfwfqmXrtKDEChHDEvoJFRHNHSRhP/IXVgY41107bZnQoVCWTpQV4zIXYMh1BwIjWc0+SdICH+fk+JBYFux0byFXjjUZI29M2w9nEtkMUbmDTM8/uw4uPr3FeQCA0m+vd6YqJI/Eju+fH7Mgbh5Lp6MEY+bKiPtedE9gRI6mVkOQuNuyJk0qeEr2fJxloBA97NlBHrIspjQJoyIHRV1bO1bT7pw/UdwsoHEoK5PDz4sIH1sFPck/gRHmG0Kubr2CFnY53dDr598wvnNEjAdw4lLMoM2uR3rVWnYAjCrFuZn9v3U9P3om3HJwe6/cNnPIKihBAc7FPdZVnMXX/5IHoAQgBTkEpGWKKTC/D2fT45oot/c5AIFuR4lmVAbvN4Jbmw5KZR7PFEi4wcKPmXKLlMXK4phfLxsthk8uICxPakhTnzjfI0P5uPd3dtaSYRTeQ2ohDXxjGRjNFt1VuUDfFI9fkt+SuNaFMx71ccPw0Y8DtovK6Fqo9Gl5ABk0CmFuSFIakL+9nbL1rBengus7KKaG0WVJU5cD0ZU0kF4Nzp5dsxJqtpXvmonBpY4VQR7fZdu2C/pGMFxObUd0dsIy7D8hFWBtqBH03i6Pwb8vBROSlwl31DVmCebbv7vkTe3Q7guTrQAHf2T63EIz/5iCmeirwjF+DpJLdLzYkjkGDNoY/FNftpuNPKHyQSL7jP2DKfKpNJdXm60CFAHyTGpzYwTH5TGXjq9cY79ffLjV27qy1k9GXU5aTurV16arZC6B2bqGrDi5p/KPceqiltsnqS63CLwdetWgJwfSklCvUaV1T

In [9]:
print(history)

[AIMessage(content=[{'type': 'text', 'text': '**CLINICAL ASSESSMENT:**\nThe patient is a 58-year-old male with a history of hypertension presenting with classic signs of Acute Coronary Syndrome (ACS). The chief complaints are acute chest pain (onset 2 hours ago) radiating to the left arm and shortness of breath. Vital signs demonstrate tachycardia (HR 105) and hypertension (145/90), consistent with sympathetic surge due to pain or cardiac distress. Most critically, the Vision Analysis of the ECG indicates possible ST elevation. This clinical constellation (symptoms + demographics + ECG findings) is highly suggestive of an acute ST-Elevation Myocardial Infarction (STEMI), which is a time-critical medical emergency.\n\n**DIFFERENTIAL DIAGNOSES:**\n1.  **Acute ST-Elevation Myocardial Infarction (STEMI)** - [Evidence: Acute chest pain radiating to left arm, onset <12 hours, tachycardia, and "Possible ST elevation" on ECG analysis.]\n2.  **Non-ST-Elevation Myocardial Infarction (NSTEMI) / U

In [12]:
msg = history[-1]  # the AIMessage object

text = msg.content[0]['text']
signature = msg.content[0]['extras']['signature']

model = msg.response_metadata['model_name']
tokens = msg.usage_metadata['total_tokens']


In [13]:
print(text)

**CLINICAL ASSESSMENT:**
The patient is a 58-year-old male with a history of hypertension presenting with classic signs of Acute Coronary Syndrome (ACS). The chief complaints are acute chest pain (onset 2 hours ago) radiating to the left arm and shortness of breath. Vital signs demonstrate tachycardia (HR 105) and hypertension (145/90), consistent with sympathetic surge due to pain or cardiac distress. Most critically, the Vision Analysis of the ECG indicates possible ST elevation. This clinical constellation (symptoms + demographics + ECG findings) is highly suggestive of an acute ST-Elevation Myocardial Infarction (STEMI), which is a time-critical medical emergency.

**DIFFERENTIAL DIAGNOSES:**
1.  **Acute ST-Elevation Myocardial Infarction (STEMI)** - [Evidence: Acute chest pain radiating to left arm, onset <12 hours, tachycardia, and "Possible ST elevation" on ECG analysis.]
2.  **Non-ST-Elevation Myocardial Infarction (NSTEMI) / Unstable Angina** - [Evidence: History of hypertensi